# 00 — Guardrails audit

Live invariant dashboard for **G1–G10 + RISK-04** plus the **runtime-adapter** seam (`hsb.runtime.{claude,codex,protocol,codex_guards}`) against the current source tree.

**No LLM, no MCP, no network.** Every cell either passes silently or raises.

If a cell fails, the guardrail it covers has regressed — open the source file referenced in the assertion and figure out which structural defense was weakened.

The notebook is **runtime-agnostic**: the same cells assert the invariants whether agents are configured for Claude or Codex. Where a guard is runtime-specific (e.g. Codex `~/.codex/config.toml`), the cell makes that explicit.

Reference: `README.md#guardrails-g1g10`, `src/hsb/agents/_sdk_options.py`, `src/hsb/agents/risk_agent.py`, `src/hsb/runtime/`, `docs/superpowers/specs/2026-05-09-codex-oauth-alt-runtime-design.md`.

In [ ]:
from _helpers import ensure_src_on_path, runtime_summary

ROOT = ensure_src_on_path()
print("repo root =", ROOT)
print("\nHSB_RUNTIME_* selection:\n" + runtime_summary())

## G1 — OAuth2 only (function-entry guard)

The guard is **function-entry**, not module-import. A dev environment with `ANTHROPIC_API_KEY` or `OPENAI_API_KEY` set should not break pytest collection — but a call to `make_options()` / `make_agent_options()` must refuse.

We probe both forbidden vars across both factories, then confirm clean construction works.

In [ ]:
import importlib
import os

# Importing the chokepoint module must NOT raise even if a metered key is set.
os.environ["ANTHROPIC_API_KEY"] = "pretend-this-is-leaked"
os.environ["OPENAI_API_KEY"] = "pretend-this-is-leaked-too"
from hsb.agents import _sdk_options

importlib.reload(_sdk_options)  # prove a reload is also safe
print("module-import path is clean even with metered keys set")

In [ ]:
# G1 must raise at function-entry of make_options() / assert_oauth2_only() /
# make_agent_options() — for either forbidden var.
import os

from hsb.agents._sdk_options import (
    assert_oauth2_only,
    make_agent_options,
    make_options,
)

for var in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY"):
    # Clean both first, then set just one to test isolation.
    os.environ.pop("ANTHROPIC_API_KEY", None)
    os.environ.pop("OPENAI_API_KEY", None)
    os.environ[var] = "leaked"

    raised = False
    try:
        assert_oauth2_only()
    except RuntimeError as e:
        raised = True
        assert "G1 violation" in str(e) and var in str(e)
    assert raised, f"G1 did not raise when {var} is set"

    raised = False
    try:
        make_options(permission_mode="acceptEdits", allowed_tools=["Read"])
    except RuntimeError:
        raised = True
    assert raised, f"make_options should refuse when {var} is set"

    raised = False
    try:
        make_agent_options(
            system_prompt="x",
            allowed_tools=["Read"],
            permission_mode="acceptEdits",
            max_turns=1,
            model="claude-haiku-4-5",
        )
    except RuntimeError:
        raised = True
    assert raised, f"make_agent_options should refuse when {var} is set"

    del os.environ[var]

print("G1: function-entry guard fires for both ANTHROPIC_API_KEY and OPENAI_API_KEY")

In [ ]:
# With the env clean, both factories must construct without complaint.
opts = make_options(permission_mode="acceptEdits", allowed_tools=["Read"])
assert "Read" in (opts.allowed_tools or [])

agent_opts = make_agent_options(
    system_prompt="x",
    allowed_tools=["Read"],
    permission_mode="acceptEdits",
    max_turns=1,
    model="claude-haiku-4-5",
)
assert agent_opts.allowed_tools == ("Read",)
print("G1: clean env path produces both ClaudeAgentOptions and AgentOptions OK")

## G2 — `"Agent"` is forbidden in any `allowed_tools`

WORC-02: no sub-subagent dispatch. Both factories raise `ValueError` if `"Agent"` slips into `allowed_tools`. Belt-and-braces: grep every agent file for an `Agent` literal in any `allowed_tools=[...]` block.

In [ ]:
raised = False
try:
    make_options(permission_mode="acceptEdits", allowed_tools=["Read", "Agent"])
except ValueError as e:
    raised = True
    assert "G2 violation" in str(e)
assert raised, "G2 should refuse Agent in allowed_tools (make_options)"

raised = False
try:
    make_agent_options(
        system_prompt="x",
        allowed_tools=["Read", "Agent"],
        permission_mode="acceptEdits",
        max_turns=1,
        model="claude-haiku-4-5",
    )
except ValueError as e:
    raised = True
    assert "G2 violation" in str(e)
assert raised, "G2 should refuse Agent in allowed_tools (make_agent_options)"
print("G2: both factories reject Agent in allowed_tools")

In [ ]:
# Belt-and-braces: grep every agent file for an 'Agent' literal in an allow-list.
import re

agent_dir = ROOT / "src" / "hsb" / "agents"
violations = []
for p in sorted(agent_dir.glob("*.py")):
    text = p.read_text()
    for m in re.finditer(r"allowed_tools\s*=\s*\[(.*?)\]", text, re.DOTALL):
        body = m.group(1)
        if re.search(r"['\"]Agent['\"]", body):
            violations.append((p.name, body[:120]))
assert not violations, f"G2 grep found Agent in an allowed_tools literal: {violations}"
print("G2: no agent module declares Agent in its allowed_tools")

## G3 — Runtime backstop catches `Task`-tool dispatch

If the SDK ever regresses and bypasses `allowed_tools`, the per-message scan in every receive loop catches a Task block and raises. G3 is Claude-specific — the message shape is a `claude_agent_sdk.AssistantMessage`. On Codex the equivalent defence is `verify_codex_mcp` + the Codex SDK's own MCP whitelist (we check that further down).

In [ ]:
from claude_agent_sdk import AssistantMessage

from hsb.agents._sdk_options import assert_no_task_dispatch


class FakeBlock:
    def __init__(self, name):
        self.name = name


msg = AssistantMessage(content=[FakeBlock("Task")], model="claude-opus-4-7")
raised = False
try:
    assert_no_task_dispatch(msg)
except RuntimeError as e:
    raised = True
    assert "G3 violation" in str(e)
assert raised, "G3 should raise on Task block"
print("G3: AssistantMessage with Task block raises")

In [ ]:
# Innocent messages must NOT raise.
msg = AssistantMessage(
    content=[FakeBlock("Read"), FakeBlock("Bash")], model="claude-opus-4-7"
)
assert_no_task_dispatch(msg)  # no exception
print("G3: clean assistant message passes through silently")

## G4 — Risk Agent skill 14 is structurally air-gapped

The Auto-Improvement Trigger SDK call has `allowed_tools=[]`, no MCP, Haiku model, and a hard budget. We verify the literals appear in the source — the structural defence, not just the runtime assertion. Risk Agent stays on Claude (HookMatcher API has no Codex equivalent), so this guard is Claude-specific by design.

In [ ]:
ra = (ROOT / "src" / "hsb" / "agents" / "risk_agent.py").read_text()
assert "allowed_tools=[]" in ra, "G4 layer 1: empty allowed_tools missing"
assert 'model="claude-haiku-4-5"' in ra or "model='claude-haiku-4-5'" in ra, (
    "G4: Haiku pin missing"
)
assert "max_budget_usd=0.05" in ra, "G4: budget cap missing"
assert "max_turns=3" in ra, "G4: max_turns cap missing"
print("G4: skill-14 air-gap config literals all present")

## RISK-04 — 4-layer defense (the strongest milestone-level invariant)

1. STRUCTURAL — skill-14 SDK call (covered by G4 above).
2. PARSE-TIME — `AutoImprovementTrigger.linear_state` is `Literal["suggested"]`.
3. IMPORT-TIME — `risk_agent.py` does NOT import `hsb.agents.linear_agent`.
4. RUNTIME — `linear_write_guard` denies frames originating in `risk_agent.py` (except the operator-delegated path).

In [ ]:
assert "from hsb.agents.linear_agent" not in ra, (
    "RISK-04 layer 3: risk_agent imports linear_agent"
)
assert "import hsb.agents.linear_agent" not in ra, (
    "RISK-04 layer 3: risk_agent imports linear_agent"
)
print("RISK-04 layer 3 (import-time): risk_agent.py does not import linear_agent")

In [ ]:
from pydantic import ValidationError

from hsb.contracts.risk import AutoImprovementTrigger

trig = AutoImprovementTrigger(
    title="t", description="d", pattern_evidence=["LIN-1", "LIN-2"], suggested_type="x"
)
assert trig.linear_state == "suggested"
raised = False
try:
    AutoImprovementTrigger(
        title="t",
        description="d",
        pattern_evidence=["LIN-1", "LIN-2"],
        suggested_type="x",
        linear_state="created",
    )
except ValidationError:
    raised = True
assert raised, "RISK-04 layer 2: parser accepted linear_state != 'suggested'"
print("RISK-04 layer 2 (parse-time): linear_state Literal enforced")

In [ ]:
# Layer 4: the linear_write_guard helper. We exercise it by calling a wrapped
# function from a 'fake risk_agent' frame and confirming PermissionError.

from hsb.agents._sdk_options import linear_write_guard


@linear_write_guard
def fake_write():
    return "ok"


# Direct call from this notebook (no risk_agent.py frame in stack) -> allowed.
assert fake_write() == "ok"
print("RISK-04 layer 4: guard allows non-risk callers")

# Simulate a call originating from risk_agent.py by exec-ing in a faked filename.
import importlib.util
import os
import tempfile

tmpdir = tempfile.mkdtemp()
fake_path = os.path.join(tmpdir, "hsb", "agents", "risk_agent.py")
os.makedirs(os.path.dirname(fake_path), exist_ok=True)
with open(fake_path, "w") as f:
    f.write("def call_it(fn):\n    return fn()\n")

spec = importlib.util.spec_from_file_location("hsb.agents.risk_agent_dummy", fake_path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
raised = False
try:
    mod.call_it(fake_write)
except PermissionError as e:
    raised = True
    assert "RISK-04" in str(e)
assert raised, "RISK-04 layer 4: guard did not deny risk_agent frame"
print("RISK-04 layer 4: guard denies frames from risk_agent.py")

## G9 — Knowledge Store pre-write hook

`KnowledgeStorageInput.applicability` rejects `"all tasks"`, `"all"`, `"n/a"`, `"tbd"`, and empty/whitespace strings (case-insensitive). Pure Pydantic — runtime-agnostic.

In [ ]:
from hsb.contracts.knowledge import KnowledgeStorageInput

base = dict(
    title="Caching DB writes",
    type="implementation",
    context="ctx",
    evidence={
        "linear_issue": "LIN-1",
        "pr": "#1",
        "files": ["x.py"],
        "qa_finding": "F1",
    },
    insight="ins",
    recommendation="rec",
    date="2026-05-09",
)
rejected = ["all tasks", "All Tasks", "ALL", "n/a", "tbd", "TBD", "", "   ", " "]
for bad in rejected:
    raised = False
    try:
        KnowledgeStorageInput(**base, applicability=bad)
    except Exception:
        raised = True
    assert raised, f"G9 accepted {bad!r}"
ok = KnowledgeStorageInput(
    **base, applicability="Postgres write paths in payments domain"
)
assert ok.applicability.startswith("Postgres")
print("G9: applicability validator rejects all banned values, accepts a real one")

## QA cycle cap (Pitfall 2 / QAAG-04)

Pydantic `model_validator` rejects `qa_cycle_count >= 3` with `qa_status='changes_required'` and requires `tech_debt_annotation` at cap. This is the deterministic last line of defence — the SKILL.md instruction is only probabilistic. Runtime-agnostic.

In [ ]:
from hsb.contracts.qa import QAOutput

# Cap reached + still changes_required -> must fail
raised = False
try:
    QAOutput(
        work_item_id="LIN-1",
        qa_status="changes_required",
        qa_cycle_count=3,
        summary="s",
        findings=[],
        tech_debt_annotation="t",
    )
except Exception:
    raised = True
assert raised, "QA cap accepted changes_required at cycle 3"

# Cap reached + approved + missing annotation -> must fail
raised = False
try:
    QAOutput(
        work_item_id="LIN-1",
        qa_status="approved",
        qa_cycle_count=3,
        summary="s",
        findings=[],
    )
except Exception:
    raised = True
assert raised, "QA cap accepted missing tech_debt_annotation at cycle 3"

# Cap reached + approved + annotation present -> OK
ok = QAOutput(
    work_item_id="LIN-1",
    qa_status="approved",
    qa_cycle_count=3,
    summary="s",
    findings=[],
    tech_debt_annotation="leaving X for follow-up",
)
assert ok.qa_cycle_count == 3
print("QA cycle cap: validator rejects both runaway shapes, accepts the canonical one")

## Runtime adapter — Protocol shape, factories, per-agent resolution

Surface added by the Codex alt-runtime work (see spec). Three structural invariants:

1. **Two implementations only**: `ClaudeRuntime` and `CodexRuntime`. Both expose a `query(prompt, options)` async iterator and a `client(options)` placeholder.
2. **Per-agent selection** via `HSB_RUNTIME_<AGENT>` (default `claude`); WIO is hard-blocked from `codex` until the stateful `ClaudeSDKClient` session has a Codex equivalent.
3. **Translation seam** — `make_agent_options()` produces a runtime-neutral `AgentOptions`; each runtime translates to its native option shape at the seam. The notebook does NOT instantiate `CodexRuntime` (which validates `~/.codex/config.toml`); we only confirm the symbols exist and `resolve_runtime` routes correctly.

In [ ]:
import hsb.runtime as runtime_pkg
from hsb.runtime import codex as codex_module
from hsb.runtime import codex_guards
from hsb.runtime.claude import ClaudeRuntime
from hsb.runtime.protocol import PermissionMode, RuntimeName

# Names the spec promises must remain on hsb.runtime — hasattr is the actual
# check (the bare `from ... import` would only fail if the names disappeared).
for sym in ("AgentOptions", "Message", "Runtime", "StatefulClient"):
    assert hasattr(runtime_pkg, sym), f"hsb.runtime missing {sym!r}"
assert PermissionMode.__args__ == (
    "default",
    "acceptEdits",
    "plan",
    "bypassPermissions",
), PermissionMode.__args__
assert RuntimeName.__args__ == ("claude", "codex"), RuntimeName.__args__
assert ClaudeRuntime().name == "claude"
# CodexRuntime constructor calls assert_codex_oauth_only() which reads
# ~/.codex/config.toml. We only confirm the class is importable + has the
# right .name attribute hint; instantiation is exercised in notebook 04/05
# where the operator opts in.
assert codex_module.CodexRuntime.name == "codex"
assert hasattr(codex_guards, "assert_codex_oauth_only")
assert hasattr(codex_guards, "verify_codex_mcp")
print(
    "runtime adapter: ClaudeRuntime + CodexRuntime + protocol + guards all importable"
)

In [ ]:
import os

from hsb.agents._sdk_options import resolve_runtime

# Default -> Claude
os.environ.pop("HSB_RUNTIME_BACKLOG", None)
rt = resolve_runtime("backlog")
assert rt.name == "claude", rt.name

# Explicit claude
os.environ["HSB_RUNTIME_BACKLOG"] = "claude"
assert resolve_runtime("backlog").name == "claude"

# Invalid value -> ValueError
os.environ["HSB_RUNTIME_BACKLOG"] = "bogus"
raised = False
try:
    resolve_runtime("backlog")
except ValueError:
    raised = True
assert raised, "resolve_runtime accepted invalid runtime name"
os.environ.pop("HSB_RUNTIME_BACKLOG", None)

# WIO hard-block: HSB_RUNTIME_WIO=codex must raise (stateful session has no Codex equivalent)
os.environ["HSB_RUNTIME_WIO"] = "codex"
raised = False
try:
    resolve_runtime("wio")
except ValueError as e:
    raised = True
    assert "WIO" in str(e) or "wio" in str(e).lower()
assert raised, "WIO hard-block missing — codex selection should raise"
os.environ.pop("HSB_RUNTIME_WIO", None)
print("resolve_runtime: default/claude/invalid all behave; WIO codex hard-block intact")

## Smoke: every agent + runtime module imports cleanly

If a structural change breaks a top-level import, every other notebook will misreport. Cheap canary.

In [ ]:
import importlib

modules = [
    "hsb.agents._sdk_options",
    "hsb.agents.hooks",
    "hsb.agents.linear_agent",
    "hsb.agents.backlog_agent",
    "hsb.agents.builder_agent",
    "hsb.agents.git_agent",
    "hsb.agents.qa_agent",
    "hsb.agents.work_item_orchestrator",
    "hsb.agents.global_orchestrator",
    "hsb.agents.main_orchestrator",
    "hsb.agents.uat_agent",
    "hsb.agents.risk_agent",
    "hsb.agents.intelligence_agent",
    "hsb.contracts.qa",
    "hsb.contracts.knowledge",
    "hsb.contracts.risk",
    "hsb.contracts.uat",
    "hsb.contracts.global_orchestrator",
    "hsb.contracts.main_orchestrator",
    "hsb.contracts.orchestrator",
    "hsb.runtime",
    "hsb.runtime.protocol",
    "hsb.runtime.claude",
    "hsb.runtime.codex",
    "hsb.runtime.codex_guards",
]
for m in modules:
    importlib.import_module(m)
print(f"OK — all {len(modules)} modules import cleanly")